## Лабораторная работа № 1
## Выполнение разведочного анализа больших данных с использованием фреймворка Apache Spark

### Часть 1. Подготовка датасета Delivery ETA

Прочитаем датасет из HDFS, проверим структуру и типы признаков, выполним преобразования и сохраним результат в Apache Iceberg.

Объект наблюдения — потенциальный заказ в сервисе онлайн-доставки продуктов. Целевая переменная `delivery_eta_minutes` — время прибытия курьера в минутах. Описание признаков — в файле `description_data_set.md`. Исходный файл `delivery_eta.parquet` занимает около 10,3 ГиБ.

Ноутбук запускается на сервере с Spark 3.5 / Scala 2.12 и YARN. До первого запуска файл нужно загрузить из локальной папки удалённого сервера в HDFS. Выполните в **терминале сервера**, находясь в папке с ноутбуками:

```bash
# Проверяем HDFS и доступное место.
hdfs dfs -ls /
hdfs dfs -df -h

# Загружаем файл из локальной папки удалённого сервера в HDFS.
hdfs dfs -mkdir -p /user/$USER/datasets
hdfs dfs -copyFromLocal delivery_eta.parquet /user/$USER/datasets/

# Проверяем результат загрузки.
hdfs dfs -ls -h /user/$USER/datasets/delivery_eta.parquet
```

Эта загрузка выполняется один раз. Если файл уже находится в HDFS по указанному адресу, повторять команды не нужно. Вторая часть читает таблицу Iceberg, сохранённую этим ноутбуком.


Подключим необходимые библиотеки. Для преобразований используем PySpark DataFrame API.


In [269]:
import os  # Имя пользователя из окружения операционной системы.
from pyspark import SparkConf  # Настройки ресурсов Spark.
from pyspark.sql import SparkSession, DataFrame  # Сессия и распределённая таблица.
from pyspark.sql import functions as F  # Функции обработки столбцов.
import pandas as pd


In [270]:
from enum import Enum

class Version(Enum):
    prod = 0
    debug = 1

Укажем адрес предварительно загруженного файла в HDFS и имена базы и таблицы. Имя базы должно совпадать с указанным во второй части. При повторной записи содержимое таблицы заменяется.


In [271]:
# Имя текущего пользователя сервера для путей HDFS.
user_name = os.getenv("USER")

# Адрес датасета, заранее загруженного в HDFS.
path = f"hdfs:///user/{user_name}/datasets/delivery_eta.parquet"

# Имена базы и таблицы для подготовленных данных.

database_name = "delivery_eta_database"
table_name = "sobd_lab1_delivery_eta"
full_table_name = f"spark_catalog.{database_name}.{table_name}"


""" const """
count_col = 10
count_row = 5
DEBUG_LIMIT_ROW=500_000
VERSION = Version.debug # todo поменять на прод. уменьшает размер df


Настроим Spark для выполнения на YARN: два исполнителя по 8 ядер и 12 ГБ памяти. Подключим Iceberg и зададим пользовательское хранилище таблиц в HDFS.


In [272]:
def create_spark_configuration() -> SparkConf:
    
    conf = SparkConf()
    conf.setAppName("lab 1 Delivery ETA")
   
    conf.setMaster("yarn")
    conf.set("spark.submit.deployMode", "client")
    conf.set("spark.executor.memory", "12g")
    # Каждому исполнителю выделяем 8 процессорных ядер.
    conf.set("spark.executor.cores", "8") # исполнителю - 8 ядер
    conf.set("spark.executor.instances", "2") #по 2 сущности исполнителя
    conf.set("spark.driver.memory", "4g")
    conf.set("spark.driver.cores", "2")
    
    conf.set("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.0")
    conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    conf.set("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkCatalog")
    conf.set("spark.sql.catalog.spark_catalog.type", "hadoop")
    conf.set("spark.sql.catalog.spark_catalog.warehouse", f"hdfs:///user/{user_name}/warehouse")
    conf.set("spark.sql.catalog.spark_catalog.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")
    return conf

conf = create_spark_configuration()


Создадим сессию Spark. Если в ядре уже работает сессия с другими настройками, сначала остановите её или перезапустите ядро.


In [273]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark


26/11/02 02:03:17 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
26/11/02 02:03:36 WARN Client: Same path resource file:///home/user0/.ivy2/jars/org.apache.iceberg_iceberg-spark-runtime-3.5_2.12-1.6.0.jar added multiple times to distributed cache.


### Чтение данных из HDFS

Прочитаем предварительно загруженный файл в датафрейм Spark. Parquet хранит названия и типы столбцов, поэтому отдельно задавать схему не требуется.

Число строк и диапазон целевой переменной вычислим ниже на загруженных данных. Короткий файл `delivery_eta.csv` — только образец строк, расчёты выполняем по полному Parquet.


In [274]:

print(VERSION)
if VERSION is Version.prod:
    df = spark.read.parquet(path)
else:
    df = spark.read.parquet(path).limit(DEBUG_LIMIT_ROW)

print(df.count())    
df.printSchema()


Version.debug


26/11/02 02:04:30 WARN YarnScheduler: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


500000
root
 |-- num_0: float (nullable = true)
 |-- num_1: float (nullable = true)
 |-- num_2: float (nullable = true)
 |-- num_3: float (nullable = true)
 |-- num_4: float (nullable = true)
 |-- num_5: float (nullable = true)
 |-- num_6: float (nullable = true)
 |-- num_7: float (nullable = true)
 |-- num_8: float (nullable = true)
 |-- num_9: float (nullable = true)
 |-- num_10: float (nullable = true)
 |-- num_11: float (nullable = true)
 |-- num_12: float (nullable = true)
 |-- num_13: float (nullable = true)
 |-- num_14: float (nullable = true)
 |-- num_15: float (nullable = true)
 |-- num_16: float (nullable = true)
 |-- num_17: float (nullable = true)
 |-- num_18: float (nullable = true)
 |-- num_19: float (nullable = true)
 |-- num_20: float (nullable = true)
 |-- num_21: float (nullable = true)
 |-- num_22: float (nullable = true)
 |-- num_23: float (nullable = true)
 |-- num_24: float (nullable = true)
 |-- num_25: float (nullable = true)
 |-- num_26: float (nullable = true)

In [297]:
from collections import defaultdict
TYPES = dict(df.dtypes)
BIN = "bin"
NUM = "num"
CAT = "cat"

def filrer_df(v: Version, dataFrame: DataFrame, count_col: int = 5)-> tuple[DataFrame, defaultdict[str, list[str]]]:
    names: defaultdict[str, list[str]] = defaultdict(list[str]) 
    for col in dataFrame.columns:
        if col.startswith("num_"):
            key = NUM
        elif col.startswith("bin_"):
            key = BIN
        elif col.startswith("cat_"):
            key = CAT
        else:
            key = "other"
        names[key].append(col)

    if v is Version.debug:
        preview_cols = names[NUM][:count_col] + names[BIN][:count_col] +  names[CAT][:count_col] + names["other"]
        dataFrame = dataFrame.select(preview_cols)
        
    return dataFrame, names


In [296]:
df, cols = filrer_df(VERSION, df, count_col=count_col)

for key, value in cols.items():
    print(f"key: {key}, value: {len(value)}")

df.limit(count_row).toPandas().head(count_row)

key: num, value: 10
key: bin, value: 1
key: cat, value: 2
key: other, value: 2


,num_0,num_1,num_2,num_3,num_4,num_5,num_6,num_7,num_8,num_9,bin_0,cat_0,cat_1,timestamp,delivery_eta_minutes
0,-0.312376,0.222201,0.239380,-3.089935,-0.312717,0.206726,0.229797,-2.615605,-0.486253,1.036617,True,310,323,2023-11-23 11:26:04,3.966667
1,0.514923,0.817902,0.553999,0.002509,0.518977,0.831198,0.578137,0.013393,-0.729960,-0.757225,True,128,129,2023-12-04 20:38:07,2.200000
2,0.478963,0.798769,0.562306,-0.226489,0.488523,0.814481,0.605163,-0.177679,-0.201070,-1.165710,True,265,276,2023-10-23 17:10:33,5.100000
3,-0.052716,0.474278,0.444534,-2.131682,-0.042444,0.483188,0.453964,-2.043681,-0.896899,-0.611855,True,56,56,2023-11-17 19:40:04,2.050000
4,-0.005018,0.346817,0.220058,-1.197193,-0.066140,0.272722,0.181429,-1.221357,0.306749,0.366357,True,240,249,2023-11-23 09:16:26,7.850000


### Содержание датасета и назначение столбцов

Каждая строка описывает потенциальный заказ в сервисе онлайн-доставки продуктов. Задача — прогнозировать, через сколько минут прибудет курьер. Признаки обезличены: они агрегируют доступность курьеров, гео- и навигационные оценки, историю предыдущих доставок и характеристики времени суток.

| Название столбца | Расшифровка |
|---|---|
| `num_0` … `num_220` | 221 обезличенный стандартизованный числовой признак. Значения обычно около нуля, во многих столбцах есть пропуски |
| `bin_0` | Обезличенный бинарный признак со значениями 0 и 1. Точная семантика не раскрыта |
| `cat_0` | Первый обезличенный категориальный признак. Значения — числовые коды категорий |
| `cat_1` | Второй обезличенный категориальный признак. Значения — числовые коды категорий |
| `cat_2` | Третий обезличенный категориальный признак. Значения — числовые коды категорий |
| `timestamp` | Дата и время формирования заказа. Используется для анализа по времени и хронологического разбиения данных |
| `delivery_eta_minutes` | Время прибытия курьера в минутах. Целевая переменная для прогнозирования |

Всего в исходном файле **227 столбцов**: 225 входных признаков, временная метка и целевая переменная.

Числовые коды `cat_*` обозначают категории, поэтому не трактуем их как количественные величины. Среднее арифметическое по коду зоны или типа заказа предметного смысла не имеет.


Выведем схему исходного датафрейма. Полный набор из 227 столбцов сохраняется, хотя для просмотра выше выбрана только часть.


In [278]:
# Печатаем схему: имена и типы столбцов. nullable = true означает, что в столбце допускаются пропуски.
df.printSchema()

root
 |-- num_0: float (nullable = true)
 |-- num_1: float (nullable = true)
 |-- num_2: float (nullable = true)
 |-- num_3: float (nullable = true)
 |-- num_4: float (nullable = true)
 |-- num_5: float (nullable = true)
 |-- num_6: float (nullable = true)
 |-- num_7: float (nullable = true)
 |-- num_8: float (nullable = true)
 |-- num_9: float (nullable = true)
 |-- bin_0: float (nullable = true)
 |-- cat_0: long (nullable = true)
 |-- cat_1: long (nullable = true)
 |-- cat_2: long (nullable = true)
 |-- timestamp: timestamp_ntz (nullable = true)
 |-- delivery_eta_minutes: float (nullable = true)



### Первичные метрики

на числовые параметры

| Название столбца | Расшифровка |
|---|---|
|count|Количество строк|
|mean|Среднее значение|
|stddev|Стандартное отклонение, насколько значения в среднем разбросаны вокруг среднего|
|min|Минимальное значение|
|25%|Персентиль 25%|
|50%|Персентиль 50%|
|75%|Персентиль 75%|
|max|Максимальное занчение|



In [279]:
n_cores = spark.sparkContext.defaultParallelism
part_number = df.rdd.getNumPartitions()
count_reparatiton = n_cores * max(part_number // part_number, 2)
print(f"Количество ядер - {n_cores}")
print(f"Количество партиций - {part_number}")
print(f"count repartition - {count_reparatiton}")

tmp_df = df.repartition(count_reparatiton) if part_number < n_cores else df

summary = tmp_df.summary() if VERSION is Version.debug else tmp_df.describe()
print("Сводная таблица")
summary.toPandas().set_index("summary").T.head(len(df.columns))

Количество ядер - 16
Количество партиций - 1
count repartition - 32
Сводная таблица


summary,count,mean,stddev,min,25%,50%,75%,max
num_0,500000,1.6642702360078693E-4,1.0107468717044386,-4.8455844,-0.67415226,0.0029273294,0.68316925,4.648853
num_1,500000,-0.0035724769941703417,1.010375928128748,-4.741838,-0.6849896,-0.0018818531,0.67475235,4.2659206
num_2,500000,-0.004011384268064517,1.0096431668615387,-4.627289,-0.69112104,-0.006272881,0.67527753,4.585457
num_3,500000,0.009653925252219663,1.0083670776689233,-3.089935,-0.6658523,0.013800685,0.6847631,5.1993375
num_4,500000,0.001532741366266353,1.0055320838015314,-4.1349034,-0.67448574,0.005326822,0.6834236,5.1993375
num_5,500000,-0.00514347969402757,1.0073005719805512,-3.8607757,-0.6890404,-0.0026681826,0.67860806,5.1993375
num_6,500000,-0.0022784249754624324,1.003109312238199,-5.1993375,-0.6867245,-0.0030895858,0.6786363,4.076813
num_7,500000,0.01004300791244139,1.001533664840533,-5.08913,-0.6648397,0.015505151,0.68346095,4.4328785
num_8,500000,-0.004197412172269771,0.9953174918078784,-3.2124383,-0.6815636,-0.0028763816,0.67265236,3.8383608
num_9,500000,0.003980738293453731,0.9995659104865983,-3.1428502,-0.6743934,0.0010029672,0.6778497,4.7507577


### `bin` признаки

проверяем на Nan & Null

Переделаем `bin` признаки в булевый параметр

In [280]:
def check_miss(df: DataFrame, col: list[str]) -> pd.DataFrame:
    types = dict(df.dtypes)
    if any(types[c] not in {"double", "float"} for c in col):
        raise ValueError("Use only `double`, `float` columns")

    # 1) выравниваем партиции ТОЛЬКО если их мало
    n_cores = df.sparkSession.sparkContext.defaultParallelism
    if df.rdd.getNumPartitions() < n_cores:
        df = df.repartition(n_cores * 2)

    # 2) все проверки в один agg — ОДИН скан вместо двух
    exprs = []
    for c in col:
        exprs.append(F.sum(F.col(c).isNull().cast("int")).alias(f"{c}__nulls"))
        exprs.append(F.sum(F.isnan(F.col(c)).cast("int")).alias(f"{c}__nans"))

    row = df.agg(*exprs).toPandas().iloc[0]

    # 3) reshape в pandas (крошечный объект, без join)
    res = pd.DataFrame(
        {
            "nans":  [row[f"{c}__nans"]  for c in col],
            "nulls": [row[f"{c}__nulls"] for c in col],
        },
        index=col,
    )
    return res


Проверка на Null & NaN

In [281]:
res = check_miss(df, cols[BIN])
res.head(len(res))

,nans,nulls
bin_0,0,0


Преобразование `bin`

In [282]:
df = df.withColumns({c: F.col(c).cast("Boolean") for c in cols[BIN]})
df.select(*cols[BIN]).printSchema()

root
 |-- bin_0: boolean (nullable = true)



### Timestamp

In [283]:
TIMESTAMP = "timestamp"
print(TYPES[TIMESTAMP])

timestamp_ntz


Смотрим данные

In [284]:
timestamp_col = df.select(TIMESTAMP)
timestamp_col.limit(count_row).toPandas().head(count_row)

,timestamp
0,2023-11-25 19:01:37
1,2023-12-12 12:12:12
2,2023-12-14 11:07:59
3,2023-10-30 09:25:34
4,2023-12-17 16:22:38


Создаём сводную таблицу

In [285]:
rng = df.agg(
    F.min(TIMESTAMP).alias("min_ts"),
    F.max(TIMESTAMP).alias("max_ts"),
    F.count(TIMESTAMP).alias("cnt"),
    F.sum(F.col(TIMESTAMP).isNull().cast("int")).alias("cout Null")
).toPandas().T
rng.rename(columns={0:"property"}).head(len(rng))

,property
min_ts,2023-10-20 01:26:40
max_ts,2024-01-07 22:52:00
cnt,500000
cout Null,0


### Delivery eta minutes

In [286]:
DEM = "delivery_eta_minutes"
print(TYPES[DEM])

float


In [287]:
dem_col = df.select(DEM)
dem_col.limit(count_row).toPandas().head(count_row)

,delivery_eta_minutes
0,1.883333
1,6.433333
2,5.066667
3,7.716667
4,17.316668


In [288]:
res = check_miss(df, [DEM])
res.rename(columns={0:"property"}).head(len(res))

,nans,nulls
delivery_eta_minutes,0,0


### `Cat` обезлеченные категории

In [289]:
for c in cols[CAT]:
    print(f"col: {c}, type: {TYPES[c]}")

col: cat_0, type: bigint
col: cat_1, type: bigint
col: cat_2, type: bigint


In [290]:
checks_null = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols[CAT]]

n_cores = df.sparkSession.sparkContext.defaultParallelism
tmp_df = df.repartition(n_cores * 2) if df.rdd.getNumPartitions() < n_cores else df
       
df = df.repartition(n_cores * 2)
nan_s = df.select(*checks_null).toPandas().T.rename(columns={0: "nulls"})
nan_s.head(len(nan_s))

,nulls
cat_0,0
cat_1,0
cat_2,0


Дубликаты

In [291]:
cond1 = F.col("cat_0").eqNullSafe(F.col("cat_1"))
cond2 = F.col("cat_1").eqNullSafe(F.col("cat_2"))
cond3 = F.col("cat_0").eqNullSafe(F.col("cat_2"))

print(df.count())

pairs = {
    "cat_0 == cat_1": cond1,
    "cat_1 == cat_2": cond2,
    "cat_0 == cat_2": cond3,
    "cat_0 == cat_1 | cat_0 == cat_2": cond1 | cond3,
    "cat_0 == cat_1 | cat_1 == cat_2": cond1 | cond2,
    "cat_0 == cat_2 | cat_1 == cat_2": cond3 | cond2,
    "any":   cond1 | cond2 | cond3,
}

df.select(*[
    F.sum(expr.cast("int")).alias(name) 
    for name, expr in pairs.items()
]).toPandas().T.rename(columns={0: "duplicate"}).head(len(pairs))

500000


,duplicate
cat_0 == cat_1,129833
cat_1 == cat_2,500000
cat_0 == cat_2,129833
cat_0 == cat_1 | cat_0 == cat_2,129833
cat_0 == cat_1 | cat_1 == cat_2,500000
cat_0 == cat_2 | cat_1 == cat_2,500000
any,500000


Удаляем дублирующие строки

In [292]:
cols[CAT] = ["cat_0", "cat_1"]
df = df.drop("cat_2")

### `Num` показатели

In [311]:
for c in cols[NUM]:
    print(f"col: {c}, type: {TYPES[c]}")

col: num_0, type: float
col: num_1, type: float
col: num_2, type: float
col: num_3, type: float
col: num_4, type: float
col: num_5, type: float
col: num_6, type: float
col: num_7, type: float
col: num_8, type: float
col: num_9, type: float


In [298]:
res = check_miss(df, cols[NUM])
res.head(len(res))

,nans,nulls
num_0,0,0
num_1,0,0
num_2,0,0
num_3,0,0
num_4,0,0
num_5,0,0
num_6,0,0
num_7,0,0
num_8,0,0
num_9,0,0


### Проверим количество строк, диапазон времени заказа и целевой переменной.


In [299]:
print(f"Количество строк: {df.count()}")

Количество строк: 500000


In [300]:
# select() выбирает результат расчёта.
# first() получает единственную строку результата, [0] — значение в ней.
first_time = df.select(F.min(TIMESTAMP)).first()[0]
last_time = df.select(F.max(TIMESTAMP)).first()[0]
print(f"Период наблюдений: {first_time} — {last_time}")

Период наблюдений: 2023-10-20 01:01:32 — 2024-01-07 22:48:50


In [301]:
min_eta = df.select(F.min(DEM)).first()[0]
max_eta = df.select(F.max(DEM)).first()[0]
print(f"Минимальное время прибытия: {min_eta}")
print(f"Максимальное время прибытия: {max_eta}")

Минимальное время прибытия: 0.01666666753590107
Максимальное время прибытия: 129.28334045410156


In [302]:
# Сначала отбираем нужные строки, затем считаем их.
negative_time_rows = df.filter(F.col(DEM) < 1)
negative_time_count = negative_time_rows.count()
print(f"Строк с временем прибытия или мгновенным временем прибытия: {negative_time_count}")

Строк с временем прибытия или мгновенным временем прибытия: 11225


Проверка на закономерность на время меньше 1 минуты

In [303]:
# проверка моей гипотезы
# переводим время в секунды и проверяем заказы меньше 30 секунд
# Мировой рекорд Усэйна Болта на дистанции 100 метров - 9,58 секунды
prop = df.filter((F.col(DEM) * 60) <= 30).select(
    (F.col(DEM) * 60).alias("seconds"),
    *[F.col(c) for c in cols[BIN]],
    *[F.col(c) for c in cols[CAT]],
)
prop.limit(20).toPandas().head(20)

,seconds,bin_0,cat_0,cat_1
0,4.0,True,251,260
1,8.0,True,103,131
2,12.0,False,56,56
3,7.0,True,426,449
4,20.0,True,253,263
5,5.0,True,92,92
6,28.0,True,225,310
7,13.0,True,186,191
8,11.0,True,272,283
9,15.0,False,186,191


In [304]:
missing_target_rows = df.filter(F.col(DEM).isNull())
missing_target_count = missing_target_rows.count()
print(f"Пропусков времени прибытия: {missing_target_count}")

Пропусков времени прибытия: 0


In [306]:
missing_timestamp_rows = df.filter(F.col(TIMESTAMP).isNull())
missing_timestamp_count = missing_timestamp_rows.count()
print(f"Пропусков временной метки: {missing_timestamp_count}")

Пропусков временной метки: 0


In [307]:
# Передаём SQL-команду Spark: создаём базу, если её ещё нет; существующая база сохраняется.
spark.sql(f"CREATE DATABASE IF NOT EXISTS spark_catalog.{database_name}")
# Выбираем базу текущей: короткие имена таблиц будут искаться в ней.
spark.catalog.setCurrentDatabase(database_name)

Запишем таблицу.


In [308]:
# writeTo() указывает, в какую таблицу записать df; using() задаёт формат Iceberg.
# createOrReplace() создаёт таблицу или заменяет существующую. Здесь реально запускаются вычисления и запись.
df.writeTo(full_table_name).using("iceberg").createOrReplace()


Проверим наличие таблицы и прочитаем небольшой фрагмент уже из Iceberg.


In [309]:
# Показываем список таблиц базы и проверяем, что наша таблица появилась.
spark.sql(f"SHOW TABLES IN spark_catalog.{database_name}").show(truncate=False)
# Читаем сохранённую таблицу по её полному имени в новый DataFrame.
saved_df = spark.table(full_table_name)

saved_df.select(df.columns).limit(count_row).toPandas().head(count_row)



+---------------------+--------------------------------+-----------+
|namespace            |tableName                       |isTemporary|
+---------------------+--------------------------------+-----------+
|delivery_eta_database|sobd_lab1_delivery_eta          |false      |
|delivery_eta_database|sobd_lab1_delivery_eta_processed|false      |
+---------------------+--------------------------------+-----------+



,num_0,num_1,num_2,num_3,num_4,num_5,num_6,num_7,num_8,num_9,bin_0,cat_0,cat_1,timestamp,delivery_eta_minutes
0,0.485069,0.702299,0.396108,-0.040157,0.499188,0.722666,0.395186,0.060138,1.297110,1.736929,True,174,179,2023-11-14 16:48:04,5.616667
1,-0.750224,-0.538947,-0.620085,-0.234219,-0.782829,-0.600270,-0.643116,-0.134432,0.608535,0.523833,True,371,390,2023-12-27 17:25:36,7.216667
2,-1.166093,-0.317650,0.046436,-0.058999,-1.112750,-0.319044,0.002114,-0.163609,1.944071,1.427953,True,302,315,2023-12-04 09:18:43,13.883333
3,-0.847356,-0.254902,-0.019656,-0.681594,-0.871266,-0.262220,-0.052904,-0.677444,-0.678064,0.669832,True,50,391,2023-12-14 15:03:36,10.883333
4,0.356159,0.560348,0.252311,-0.299230,0.393422,0.594086,0.299042,-0.219060,0.319795,-1.409527,True,394,415,2023-12-26 12:38:45,11.600000


После завершения работы остановим сессию Spark.


In [310]:
spark.stop()